# ST-GCS with cubic Bezier reservation per region

Interactive companion to `test_bezier_order4_demo.py` at the repo root -- same
5-region "arch" corridor, same `order=2` (today's linear-segment GCS) vs `order=4`
(cubic Bezier per GCS vertex) comparison, but rendered with Plotly so the plots can
be rotated, panned, zoomed, and hovered instead of being static images.

Background: `BLACKMAGIC/AGENT.md` (the design doc for this representation) and
`BLACKMAGIC/bspline-migration-plan.md` (how it maps onto this codebase's actual
`stgcs/` module). The short version: a GCS vertex already corresponds to exactly one
region traversal with its own private decision variables, so raising the per-vertex
control-point count from 2 (`order=2`, a line segment) to 4 (`order=4`, a cubic
Bezier) makes each vertex own a Bezier segment outright -- continuity between
consecutive segments (C0 position/time, C1 velocity) is then a small set of linear
equality constraints on raw control-point differences, solved inside the same convex
program, with no separate curve-extraction step and no bilinear terms.

In [4]:
import os
import sys
from pathlib import Path

import numpy as np
from scipy.spatial import ConvexHull, QhullError
from pydrake.all import VPolytope
import plotly.graph_objects as go
from plotly.subplots import make_subplots

os.environ.setdefault("MPLCONFIGDIR", "/tmp/mpl")
os.environ.setdefault("MOSEKLM_LICENSE_FILE", str(Path.home() / "mosek" / "mosek.lic"))

# the kernel's cwd is this notebook's own directory (notebooks/), not the repo root
repo_root = Path.cwd() if (Path.cwd() / "stgcs").exists() else Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from stgcs.stgcs import STGCS
from stgcs.interval import Interval
from stgcs.geometry_utils import make_hpolytope, time_extruded, hpoly_to_vrep
from stgcs.spatialtemporal_planner import MPQuery
from stgcs.gcs.solver import solve
import stgcs.ecd as ecd_mod

## Scenario: a 5-region "arch" corridor

Start and goal sit at the *same height*, but the corridor of 5 alternating
horizontal/vertical strips rises through the middle and comes back down -- no single
straight line can satisfy it, so (almost) every one of the 4 region-to-region joints
forces a genuine bend. That gives `order=2` a visibly kinked polyline and `order=4` a
smooth arc to compare it against, across more than one boundary.

In [5]:
REGIONS = [
    np.array([[0.0, 0.0], [2.0, 0.0], [2.0, 1.0], [0.0, 1.0]]),  # v0: horizontal (start)
    np.array([[1.0, 0.0], [2.0, 0.0], [2.0, 3.0], [1.0, 3.0]]),  # v1: vertical, up
    np.array([[1.0, 2.0], [4.0, 2.0], [4.0, 3.0], [1.0, 3.0]]),  # v2: horizontal, across the top
    np.array([[3.0, 0.0], [4.0, 0.0], [4.0, 3.0], [3.0, 3.0]]),  # v3: vertical, back down
    np.array([[3.0, 0.0], [5.0, 0.0], [5.0, 1.0], [3.0, 1.0]]),  # v4: horizontal (goal)
]
START = np.array([0.2, 0.5])   # lives only in v0
GOAL = np.array([4.5, 0.5])    # lives only in v4, same height as start
VLIMIT = 2.0
TMAX = 30.0
DT = 0.05  # for order > 2, floors *each* consecutive control-point pair (not just
           # the whole segment, as it does for order == 2) -- keeps interior control-
           # point spacing well-conditioned instead of letting the solver collapse it.

# named plotly-recognized hex colors (the matplotlib "tab10" palette), reused for
# both the region shading and the per-segment hull coloring below.
REGION_COLORS = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
    "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf",
]


def build_stgcs(order: int) -> STGCS:
    stgcs = STGCS([REGIONS[0]], t0=0.0, tmax=TMAX, vlimit=VLIMIT, dt=DT, order=order)
    names = [f"v{i}" for i in range(len(REGIONS))]
    for name, region in zip(names, REGIONS):
        hpoly = time_extruded(make_hpolytope(region), 0.0, TMAX)
        stgcs.add_vertex(hpoly, Interval(0.0, TMAX), name=name)
    for tail, head in zip(names[:-1], names[1:]):
        stgcs.add_edges_bidir(tail, head)
    return stgcs


def solve_query(stgcs: STGCS, order: int):
    mp_query = MPQuery(start=START, goal=GOAL, t_start=0.0, is_stay=False, vlimit=stgcs.vlimit)
    gcs_instance = stgcs.get_gcs_instance(mp_query)
    assert gcs_instance is not None, "failed to build GCS instance"
    try:
        sol = solve(gcs_instance, order=order)
    finally:
        gcs_instance.cleanup()
        stgcs.clear_query_vertices()
    assert sol is not None, "solve returned no solution"
    return sol

## Solve both representations and check continuity at every boundary

`order=2`'s kink angle should be nonzero at most joints; `order=4`'s C1 mismatch --
the raw space-time control-point difference on each side of a boundary, which F3
constrains to be exactly equal -- should read ~0 everywhere (solver tolerance).

In [6]:
sol2 = solve_query(build_stgcs(order=2), order=2)
sol4 = solve_query(build_stgcs(order=4), order=4)
num_boundaries = sol2.size - 1
assert sol4.size - 1 == num_boundaries, "order=2 and order=4 took different vertex paths"

print(f"order=2: duration={sol2.duration:.3f}s over {sol2.size} regions")
print(f"order=4: duration={sol4.duration:.3f}s over {sol4.size} regions")

for i in range(num_boundaries):
    dir_in = sol2.xB(i)[:2] - sol2.xA(i)[:2]
    dir_out = sol2.xB(i + 1)[:2] - sol2.xA(i + 1)[:2]
    cos_kink = np.dot(dir_in, dir_out) / (np.linalg.norm(dir_in) * np.linalg.norm(dir_out))
    kink_deg = np.degrees(np.arccos(np.clip(cos_kink, -1, 1)))

    tail_diff = sol4.control_point(i, sol4.order - 1) - sol4.control_point(i, sol4.order - 2)
    head_diff = sol4.control_point(i + 1, 1) - sol4.control_point(i + 1, 0)
    mismatch = np.linalg.norm(tail_diff - head_diff)

    print(f"boundary v{i}-v{i+1}: order=2 kink={kink_deg:5.1f} deg   order=4 C1 mismatch={mismatch:.2e}")

INFO:drake:Solved GCS shortest path using Mosek with convex_relaxation=true and preprocessing=false and rounding.
INFO:drake:Found 1 unique paths, discarded 999 duplicate paths.
INFO:drake:Finished 1 rounding solutions with Mosek.
INFO:drake:Solved GCS shortest path using Mosek with convex_relaxation=true and preprocessing=false and rounding.
INFO:drake:Found 1 unique paths, discarded 999 duplicate paths.
INFO:drake:Finished 1 rounding solutions with Mosek.


order=2: duration=2.150s over 5 regions
order=4: duration=2.756s over 5 regions
boundary v0-v1: order=2 kink= 13.0 deg   order=4 C1 mismatch=4.45e-09
boundary v1-v2: order=2 kink= 45.0 deg   order=4 C1 mismatch=1.44e-14
boundary v2-v3: order=2 kink= 45.0 deg   order=4 C1 mismatch=1.49e-14
boundary v3-v4: order=2 kink=  0.0 deg   order=4 C1 mismatch=8.44e-14


## Interactive space-time (x, y, t) view

This is the actual ambient space GCS solves in -- regions are time-extruded prisms in
R^3, not 2D shapes with a separate clock (see the earlier discussion in this session:
time is just the 3rd coordinate, per F4). **Drag to rotate, scroll to zoom, click a
legend entry to toggle it, hover a point for its (x, y, t).**

Includes each `order=4` segment's own space-time convex hull -- the F1 certificate
that the curve lies inside it, continuously, for the whole segment, with no sampling
required. In 2D+time these are generically full-dimensional tetrahedra (F7).

In [7]:
# union of both solutions' time span in region idx, so both curves fit in the drawn
# space-time prism regardless of their (slightly different) time allocation
def region_time_bounds(idx: int) -> tuple:
    t_lo = min(sol2.xA(idx)[-1], sol4.control_point(idx, 0)[-1])
    t_hi = max(sol2.xB(idx)[-1], sol4.control_point(idx, sol4.order - 1)[-1])
    return t_lo, t_hi


# convex hull of a raw point cloud as a Mesh3d trace (same ConvexHull-based pattern
# environment/env.py uses for its own plotly convex-set rendering). legendgroup/
# legendgrouptitle default to None (no grouping) so existing callers are unaffected.
def hull_mesh3d(vertices, color, opacity, name, showlegend=True, legendgroup=None, legendgrouptitle=None):
    vertices = np.asarray(vertices, dtype=float)
    hull = ConvexHull(vertices)
    simplices = hull.simplices
    kwargs = dict(
        x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2],
        i=simplices[:, 0], j=simplices[:, 1], k=simplices[:, 2],
        color=color, opacity=opacity, name=name, showlegend=showlegend,
        hoverinfo="name",
    )
    if legendgroup is not None:
        kwargs["legendgroup"] = legendgroup
    if legendgrouptitle is not None:
        kwargs["legendgrouptitle_text"] = legendgrouptitle
    return go.Mesh3d(**kwargs)


fig3d = go.Figure()

for i, region in enumerate(REGIONS):
    t_lo, t_hi = region_time_bounds(i)
    st_hpoly = time_extruded(make_hpolytope(region), t_lo, t_hi)
    verts = hpoly_to_vrep(st_hpoly)
    fig3d.add_trace(hull_mesh3d(verts, REGION_COLORS[i % len(REGION_COLORS)], 0.15, f"region v{i}"))

order2_xyt = np.array([sol2.xA(0)] + [sol2.xB(i) for i in range(sol2.size)])
fig3d.add_trace(go.Scatter3d(
    x=order2_xyt[:, 0], y=order2_xyt[:, 1], z=order2_xyt[:, 2],
    mode="lines+markers", line=dict(color="red", dash="dash", width=4),
    marker=dict(size=4, color="red"), name="order=2 (kinked)",
    hovertemplate="x=%{x:.2f}<br>y=%{y:.2f}<br>t=%{z:.2f}<extra>order=2</extra>",
))

ts = np.linspace(sol4.x0[-1], sol4.xT[-1], 400)
order4_xyt = np.array([sol4.lerp(t) for t in ts])
fig3d.add_trace(go.Scatter3d(
    x=order4_xyt[:, 0], y=order4_xyt[:, 1], z=order4_xyt[:, 2],
    mode="lines", line=dict(color="green", width=5), name="order=4 (C1 curve)",
    hovertemplate="x=%{x:.2f}<br>y=%{y:.2f}<br>t=%{z:.2f}<extra>order=4</extra>",
))

for idx in range(sol4.size):
    cps = sol4.control_points(idx)
    fig3d.add_trace(hull_mesh3d(cps, "green", 0.25, "order=4 hull (F1)", showlegend=(idx == 0)))
    fig3d.add_trace(go.Scatter3d(
        x=cps[:, 0], y=cps[:, 1], z=cps[:, 2], mode="markers",
        marker=dict(size=3, color="black"), name="control points", showlegend=(idx == 0),
        hovertemplate="control point<br>x=%{x:.2f}<br>y=%{y:.2f}<br>t=%{z:.2f}<extra></extra>",
    ))

fig3d.add_trace(go.Scatter3d(x=[sol2.x0[0]], y=[sol2.x0[1]], z=[sol2.x0[2]], mode="markers",
    marker=dict(size=6, color="black", symbol="diamond"), name="start"))
fig3d.add_trace(go.Scatter3d(x=[sol2.xT[0]], y=[sol2.xT[1]], z=[sol2.xT[2]], mode="markers",
    marker=dict(size=6, color="black", symbol="cross"), name="goal (order=2 arrival)"))
fig3d.add_trace(go.Scatter3d(x=[sol4.xT[0]], y=[sol4.xT[1]], z=[sol4.xT[2]], mode="markers",
    marker=dict(size=7, color="black", symbol="diamond-open"), name="goal (order=4 arrival)"))

fig3d.update_layout(
    title="space-time (x, y, t): drag to rotate, scroll to zoom, click legend to toggle",
    scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="t"),
    width=850, height=650, margin=dict(l=0, r=0, t=40, b=0),
)
fig3d

## Interactive top-down (x, y) view

Same scene, projected flat, including each `order=4` segment's spatial hull (the F1
hull projected into (x, y) -- generally thinner than the space-time version, since
curvature here is concentrated at the region joints rather than spread across a
segment's interior). **Scroll to zoom, drag to pan, click legend to toggle.**

In [8]:
fig2d = go.Figure()

for i, region in enumerate(REGIONS):
    poly = np.vstack([region, region[0]])
    fig2d.add_trace(go.Scatter(
        x=poly[:, 0], y=poly[:, 1], fill="toself",
        fillcolor=REGION_COLORS[i % len(REGION_COLORS)], opacity=0.15,
        line=dict(color=REGION_COLORS[i % len(REGION_COLORS)], width=1),
        name=f"region v{i}", hoverinfo="name",
    ))

for idx in range(sol4.size):
    cps2 = sol4.control_points(idx)[:, :2]
    hull = ConvexHull(cps2)
    loop = np.vstack([cps2[hull.vertices], cps2[hull.vertices][0]])
    fig2d.add_trace(go.Scatter(
        x=loop[:, 0], y=loop[:, 1], fill="toself",
        fillcolor="green", opacity=0.2, line=dict(color="darkgreen", width=1),
        name="order=4 per-segment hull (F1)", showlegend=(idx == 0), hoverinfo="skip",
    ))

order2_xy = np.array([sol2.xA(0)[:2]] + [sol2.xB(i)[:2] for i in range(sol2.size)])
fig2d.add_trace(go.Scatter(x=order2_xy[:, 0], y=order2_xy[:, 1], mode="lines+markers",
    line=dict(color="red", dash="dash", width=3), marker=dict(size=6, color="red"),
    name="order=2 (kinked)"))

ts = np.linspace(sol4.x0[-1], sol4.xT[-1], 400)
order4_xy = np.array([sol4.lerp(t)[:2] for t in ts])
fig2d.add_trace(go.Scatter(x=order4_xy[:, 0], y=order4_xy[:, 1], mode="lines",
    line=dict(color="green", width=3), name="order=4 (C1 curve)"))

for idx in range(sol4.size):
    cps2 = sol4.control_points(idx)[:, :2]
    fig2d.add_trace(go.Scatter(x=cps2[:, 0], y=cps2[:, 1], mode="markers+lines",
        line=dict(color="gray", dash="dot", width=1), marker=dict(size=5, color="gray"),
        name="control polygon", showlegend=(idx == 0), hoverinfo="skip"))

fig2d.add_trace(go.Scatter(x=[START[0]], y=[START[1]], mode="markers",
    marker=dict(size=14, color="black", symbol="triangle-up"), name="start"))
fig2d.add_trace(go.Scatter(x=[GOAL[0]], y=[GOAL[1]], mode="markers",
    marker=dict(size=14, color="black", symbol="star"), name="goal"))

fig2d.update_layout(
    title="top-down (x, y) projection -- scroll to zoom, drag to pan, click legend to toggle",
    xaxis_title="x", yaxis_title="y",
    yaxis=dict(scaleanchor="x", scaleratio=1),
    width=800, height=600,
)
fig2d

## Per-segment convex hulls, individually zoomed

The same F1 hulls as above, but each in its own independently-scaled subplot rather
than all sharing the top-down view's scale -- a segment that's locally near-straight
(most of them here; curvature concentrates at the joins, not mid-segment) would
otherwise show up as an imperceptible sliver. Each panel is still independently
zoomable/pannable.

In [9]:
fig_hulls = make_subplots(rows=1, cols=sol4.size, subplot_titles=[f"v{i}" for i in range(sol4.size)])

for idx in range(sol4.size):
    cps2 = sol4.control_points(idx)[:, :2]
    hull = ConvexHull(cps2)
    loop = np.vstack([cps2[hull.vertices], cps2[hull.vertices][0]])
    color = REGION_COLORS[idx % len(REGION_COLORS)]
    fig_hulls.add_trace(go.Scatter(x=loop[:, 0], y=loop[:, 1], fill="toself",
        fillcolor=color, opacity=0.35, line=dict(color="black", width=1),
        showlegend=False, hoverinfo="skip"), row=1, col=idx + 1)
    fig_hulls.add_trace(go.Scatter(x=cps2[:, 0], y=cps2[:, 1], mode="markers",
        marker=dict(color="black", size=5), showlegend=False,
        hovertemplate="x=%{x:.3f}<br>y=%{y:.3f}<extra></extra>"), row=1, col=idx + 1)
    t_lo, t_hi = sol4.control_points(idx)[0, -1], sol4.control_points(idx)[-1, -1]
    ts_seg = np.linspace(t_lo, t_hi, 60)
    curve_seg = np.array([sol4.lerp(t)[:2] for t in ts_seg])
    fig_hulls.add_trace(go.Scatter(x=curve_seg[:, 0], y=curve_seg[:, 1], mode="lines",
        line=dict(color="green", width=3), showlegend=False, hoverinfo="skip"), row=1, col=idx + 1)
    suffix = "" if idx == 0 else str(idx + 1)
    fig_hulls.update_xaxes(scaleanchor=f"y{suffix}", row=1, col=idx + 1)

fig_hulls.update_layout(
    title="order=4 per-segment convex hulls (F1 certificate)",
    height=320, width=1500, showlegend=False,
)
fig_hulls

## Minkowski inflation by an axis-aligned box footprint

Everything above used the *raw* hulls -- no robot footprint. To turn a hull into a
real reservation obstacle you inflate it by the robot's footprint (AGENT.md step 4):
`inflate(hull, footprint, margin) -> HPolytope`, via a V-rep Minkowski sum (sum the
hull's vertices with the footprint's vertices pairwise, then re-hull -- this is exact
for polytope-polytope sums: `conv(A) + conv(B) == conv(A + B)`).

The footprint here is a small axis-aligned box in (x, y), lifted into space-time with
zero time-extent (a footprint doesn't occupy a time interval, only a spatial one) --
`[[hx, hy, 0], [hx, -hy, 0], [-hx, hy, 0], [-hx, -hy, 0]]`. Applying the *same*
generic sum-then-rehull code to both `order=2`'s 2-point segment and `order=4`'s
4-point hull keeps the comparison apples-to-apples on method, continuing the
half-plane-count question from before -- but with a real twist once inflation is
actually in the picture (see the facet counts below).

In [10]:
HX, HY = 0.12, 0.08  # footprint half-extents
footprint_3d = np.array([[HX, HY, 0.0], [HX, -HY, 0.0], [-HX, HY, 0.0], [-HX, -HY, 0.0]])
footprint_2d = footprint_3d[:, :2]


def minkowski_sum(vertices_a, vertices_b):
    va = np.asarray(vertices_a, dtype=float)
    vb = np.asarray(vertices_b, dtype=float)
    return (va[:, None, :] + vb[None, :, :]).reshape(-1, va.shape[-1])


print(f"{'segment':>8} {'order=2 raw':>12} {'order=2 inflated':>17} {'order=4 raw':>12} {'order=4 inflated':>17}")
inflated_hulls_2, inflated_hulls_4 = [], []
for idx in range(sol4.size):
    cps2_raw = np.array([sol2.xA(idx), sol2.xB(idx)])
    cps4_raw = sol4.control_points(idx)
    cps2_inflated = minkowski_sum(cps2_raw, footprint_3d)
    cps4_inflated = minkowski_sum(cps4_raw, footprint_3d)
    inflated_hulls_2.append(cps2_inflated)
    inflated_hulls_4.append(cps4_inflated)

    f2r = 0  # a bare 2-point segment has no volume/facets before inflation
    f2i = ConvexHull(cps2_inflated).equations.shape[0]
    f4r = ConvexHull(cps4_raw).equations.shape[0]
    f4i = ConvexHull(cps4_inflated).equations.shape[0]
    print(f"{f'v{idx}':>8} {f2r:>12} {f2i:>17} {f4r:>12} {f4i:>17}")

 segment  order=2 raw  order=2 inflated  order=4 raw  order=4 inflated
      v0            0                12            4                24
      v1            0                12            4                24
      v2            0                12            4                24
      v3            0                12            4                20
      v4            0                12            4                20


The raw-hull result from before (`order=4`'s 4 facets < `order=2`'s 6) does **not**
carry over once real footprint inflation enters the picture this way -- `order=4`'s
inflated hulls come out to ~20-24 facets, well above `order=2`'s ~12. That's not a
contradiction, it's a difference in *method*: `stgcs/ecd.py`'s actual
`parallelotope_side_halfspace_kd` doesn't do a generic V-rep sum for the linear-
segment case -- it builds the tube in a frame aligned with the segment's own
direction, which is exactly why it lands at a fixed `2d+2 = 6` regardless of
footprint size. A single cubic Bezier segment has no one privileged direction to
align a box against the same way, so the generic (but exact) sum-then-rehull used
here doesn't get that shortcut. Closing that gap -- an analogous direction-aware
inflation for curved segments -- is real follow-up work (AGENT.md step 4 flags
`inflate` as a hook, not a solved problem), not something this notebook invents.

### Inflated hulls, space-time

Same space-time view as before, with the inflated (footprint-swept) hulls added for
both representations -- `order=2`'s swept tube in red/orange, `order=4`'s swept hull
in green, each still toggleable via the legend.

In [11]:
fig3d_inflated = go.Figure()

for i, region in enumerate(REGIONS):
    t_lo, t_hi = region_time_bounds(i)
    st_hpoly = time_extruded(make_hpolytope(region), t_lo, t_hi)
    verts = hpoly_to_vrep(st_hpoly)
    fig3d_inflated.add_trace(hull_mesh3d(verts, REGION_COLORS[i % len(REGION_COLORS)], 0.08, f"region v{i}", showlegend=(i == 0)))

fig3d_inflated.add_trace(go.Scatter3d(
    x=order2_xyt[:, 0], y=order2_xyt[:, 1], z=order2_xyt[:, 2],
    mode="lines", line=dict(color="red", dash="dash", width=4), name="order=2 (kinked)",
))
fig3d_inflated.add_trace(go.Scatter3d(
    x=order4_xyt[:, 0], y=order4_xyt[:, 1], z=order4_xyt[:, 2],
    mode="lines", line=dict(color="green", width=5), name="order=4 (C1 curve)",
))

for idx in range(sol4.size):
    fig3d_inflated.add_trace(hull_mesh3d(
        inflated_hulls_2[idx], "orange", 0.25, "order=2 inflated tube", showlegend=(idx == 0)))
    fig3d_inflated.add_trace(hull_mesh3d(
        inflated_hulls_4[idx], "green", 0.25, "order=4 inflated hull", showlegend=(idx == 0)))

fig3d_inflated.update_layout(
    title="inflated (footprint-swept) hulls in space-time",
    scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="t"),
    width=850, height=650, margin=dict(l=0, r=0, t=40, b=0),
)
fig3d_inflated

### Inflated hulls, per segment, top-down

Raw hull (thin outline) vs. inflated hull (filled) for `order=4`, one panel per
segment -- the footprint visibly thickens what were near-degenerate slivers earlier
into hulls with real width in every direction.

In [12]:
fig_hulls_inflated = make_subplots(rows=1, cols=sol4.size, subplot_titles=[f"v{i}" for i in range(sol4.size)])

for idx in range(sol4.size):
    cps2_raw = sol4.control_points(idx)[:, :2]
    raw_hull = ConvexHull(cps2_raw)
    raw_loop = np.vstack([cps2_raw[raw_hull.vertices], cps2_raw[raw_hull.vertices][0]])

    cps2_inflated = minkowski_sum(cps2_raw, footprint_2d)
    inf_hull = ConvexHull(cps2_inflated)
    inf_loop = np.vstack([cps2_inflated[inf_hull.vertices], cps2_inflated[inf_hull.vertices][0]])

    color = REGION_COLORS[idx % len(REGION_COLORS)]
    fig_hulls_inflated.add_trace(go.Scatter(x=inf_loop[:, 0], y=inf_loop[:, 1], fill="toself",
        fillcolor=color, opacity=0.35, line=dict(color="black", width=1),
        showlegend=False, hoverinfo="skip"), row=1, col=idx + 1)
    fig_hulls_inflated.add_trace(go.Scatter(x=raw_loop[:, 0], y=raw_loop[:, 1], mode="lines",
        line=dict(color="black", width=2, dash="dot"), showlegend=False, hoverinfo="skip"), row=1, col=idx + 1)
    suffix = "" if idx == 0 else str(idx + 1)
    fig_hulls_inflated.update_xaxes(scaleanchor=f"y{suffix}", row=1, col=idx + 1)

fig_hulls_inflated.update_layout(
    title="order=4: raw hull (dotted outline) vs. footprint-inflated hull (filled)",
    height=320, width=1500, showlegend=False,
)
fig_hulls_inflated

## Reserved space-time volume: which method reserves more?

This is the question that actually matters for prioritized planning: whatever a
higher-priority robot reserves gets subtracted from the graph before the next robot
plans (`stgcs/ecd.py`'s `reserve`), so a *larger* reserved volume leaves *less* free
space-time behind for whoever plans next. Facet count (above) is about how expensive
the reservation is to represent/subtract; volume is about how much it actually costs
the next robot.

We split volume into two factors that multiply together for a swept tube: the
*(x, y) footprint area* swept (spatial room needed to round the corners) and the
*duration* spent occupying it (how long the region is blocked). That split says
whether extra reserved volume comes from genuinely needing more lateral room, or
just from dwelling longer -- which, given the duration discussion earlier in this
notebook, is worth checking directly rather than assuming.

In [13]:
print(f"{'seg':>4} {'order=2 vol':>12} {'order=4 vol':>12} {'ratio':>7} {'ord2 xy-area':>13} {'ord4 xy-area':>13}")
total_vol_2 = total_vol_4 = 0.0
total_xy_2 = total_xy_4 = 0.0
per_segment_volumes = []
for idx in range(sol4.size):
    cps2_raw = np.array([sol2.xA(idx), sol2.xB(idx)])
    cps4_raw = sol4.control_points(idx)

    vol2 = ConvexHull(minkowski_sum(cps2_raw, footprint_3d)).volume
    vol4 = ConvexHull(minkowski_sum(cps4_raw, footprint_3d)).volume
    xy2 = ConvexHull(minkowski_sum(cps2_raw[:, :2], footprint_2d)).volume  # 2D hull "volume" == area
    xy4 = ConvexHull(minkowski_sum(cps4_raw[:, :2], footprint_2d)).volume

    total_vol_2 += vol2; total_vol_4 += vol4
    total_xy_2 += xy2; total_xy_4 += xy4
    per_segment_volumes.append((vol2, vol4))
    print(f"v{idx:<3} {vol2:>12.5f} {vol4:>12.5f} {vol4/vol2:>7.3f} {xy2:>13.5f} {xy4:>13.5f}")

print(f"{'TOT':>4} {total_vol_2:>12.5f} {total_vol_4:>12.5f} {total_vol_4/total_vol_2:>7.3f} {total_xy_2:>13.5f} {total_xy_4:>13.5f}")

duration_ratio = sol4.duration / sol2.duration
area_ratio = total_xy_4 / total_xy_2
volume_ratio = total_vol_4 / total_vol_2
print()
print(f"duration ratio (order=4 / order=2):        {duration_ratio:.3f}")
print(f"x,y-area ratio (order=4 / order=2):         {area_ratio:.3f}")
print(f"duration x area (predicted volume ratio):   {duration_ratio * area_ratio:.3f}")
print(f"actual volume ratio:                        {volume_ratio:.3f}")
print(f"residual (looser-inflation effect):         {volume_ratio / (duration_ratio * area_ratio):.3f}")

 seg  order=2 vol  order=4 vol   ratio  ord2 xy-area  ord4 xy-area
v0        0.01536      0.01898   1.236       0.28640       0.29229
v1        0.01920      0.03214   1.674       0.43840       0.47140
v2        0.01920      0.02448   1.275       0.19840       0.24416
v3        0.01920      0.03114   1.622       0.43840       0.46449
v4        0.00960      0.01364   1.421       0.23840       0.23888
 TOT      0.08256      0.12039   1.458       1.60000       1.71122

duration ratio (order=4 / order=2):        1.282
x,y-area ratio (order=4 / order=2):         1.070
duration x area (predicted volume ratio):   1.371
actual volume ratio:                        1.458
residual (looser-inflation effect):         1.064


Read the last block top to bottom: `duration ratio` and `x,y-area ratio` are the two
factors; their product is what you'd *predict* the volume ratio to be if reserved
volume were exactly `area x duration`; `actual volume ratio` is what the hulls
actually come out to; and `residual` is whatever's left over.

That residual (~1.06, i.e. ~6%) is real -- the same "no segment-direction shortcut
for a curved segment" effect flagged in the facet-count section -- but it's a much
smaller effect on *volume* than it was on *facet count* (order=4's naive inflation
had roughly 2x the facets of order=2's direction-aligned one, not 1.06x). That's a
useful distinction in its own right: the extra facets from a generic (non-direction-
aligned) inflation are mostly redundant-ish/oblique cuts near the hull's corners,
not bulk extra volume -- "looser" mainly costs representation size (more half-planes
to carry through ECD), only a little of it costs actual reserved space. The bulk of
the volume gap (duration x area ~= 1.37 of the observed 1.46) is duration, and
almost none of it (area ratio ~= 1.07) is genuine extra lateral room needed to round
the corners.

In [14]:
seg_labels = [f"v{i}" for i in range(sol4.size)]
vol2_list = [v[0] for v in per_segment_volumes]
vol4_list = [v[1] for v in per_segment_volumes]

fig_volumes = go.Figure()
fig_volumes.add_trace(go.Bar(x=seg_labels, y=vol2_list, name="order=2 reserved volume", marker_color="red", opacity=0.7))
fig_volumes.add_trace(go.Bar(x=seg_labels, y=vol4_list, name="order=4 reserved volume", marker_color="green", opacity=0.7))
fig_volumes.update_layout(
    title="reserved space-time volume per segment (hover for exact values)",
    xaxis_title="segment", yaxis_title="volume (space^2 x time)",
    barmode="group", width=800, height=450,
)
fig_volumes

## Exact Convex Decomposition (ECD): fragmenting the graph against a reservation

Everything above treated the 5 regions as fixed. In prioritized planning, once a
robot's trajectory is reserved, every region it touches gets sliced (`stgcs/ecd.py`'s
`slice()`/`_apply_ecd_pairs`) into the convex leftover free-space pieces -- that's
what a *second* robot's STGCS graph actually looks like. This runs the real pipeline
(not a re-implementation) end to end: `generate_all_ECD_pairs_spline` builds one
`ECDPair` per `order=4` segment (Minkowski-inflated by the same box footprint used
above) plus "parking" pairs for staying at the start/end, then `_apply_ecd_pairs`
slices all 5 regions against them.

One gotcha hit along the way, worth calling out rather than silently working around:
`build_stgcs()` (used everywhere above) passes explicit `name="v0".."v4"` to
`add_vertex`, which never advances `STGCS._next_vertex_index` (that counter only
bumps when `name=None`). `_apply_ecd_pairs` later auto-names split pieces starting
from `"v0"` again, colliding with the original name still sitting in
`stgcs._st_vertices` (`remove_vertex_from_graph` only removes it from the graph, not
that dict, by its own docstring). `build_stgcs_autoname` below sidesteps it by letting
`add_vertex` assign names itself -- it produces the identical `v0..v4` sequence, just
advances the counter correctly first.

In [15]:
def build_stgcs_autoname(order: int):
    s = STGCS([REGIONS[0]], t0=0.0, tmax=TMAX, vlimit=VLIMIT, dt=DT, order=order)
    names = []
    for region in REGIONS:
        hpoly = time_extruded(make_hpolytope(region), 0.0, TMAX)
        v = s.add_vertex(hpoly, Interval(0.0, TMAX))
        names.append(v.name)
    for tail, head in zip(names[:-1], names[1:]):
        s.add_edges_bidir(tail, head)
    return s, names


fresh, region_names = build_stgcs_autoname(order=4)
assert region_names == sol4.vertex_path

ecd_pairs = ecd_mod.generate_all_ECD_pairs_spline(
    fresh, sol4, footprint_3d, staying_tmin=0.0, staying_tmax=TMAX,
)
fragmented = ecd_mod._apply_ecd_pairs(fresh, ecd_pairs)
print(f"{len(list(fragmented.G.nodes))} pieces after slicing all 5 regions "
      f"against order=4's full reservation ({len(ecd_pairs)} ECDPairs: "
      f"{sol4.size} moving segments + parking)")

135 pieces after slicing all 5 regions against order=4's full reservation (6 ECDPairs: 5 moving segments + parking)


### Why you can't see all the pieces in one view

The pieces span an enormous range of sizes -- the leftover free time before the
reservation starts / after it ends can be huge, while facet-peeling produces slivers
down near machine precision. No single plot can show both at a sensible scale, and
the smallest ones are geometrically real but not "solids" worth drawing. Filter down
to a band that's actually visualizable before plotting anything.

In [16]:
def piece_verts(hpoly):
    return np.asarray(VPolytope(hpoly).vertices()).T


def safe_hull(points):
    try:
        return ConvexHull(points)
    except QhullError:
        return None


def piece_volume(hpoly):
    h = safe_hull(piece_verts(hpoly))
    return h.volume if h is not None else 0.0


VOL_LO, VOL_HI = 1e-4, 1.0  # excludes degenerate slivers and huge before/after leftovers

groups = {}
for name in fragmented.G.nodes:
    v = fragmented.get_vertex(name)
    root = v.root_name[0] if v.parent is not None else name
    groups.setdefault(root, []).append(v)

survivors = {}
n_total = n_tiny = n_huge = 0
for root_name in sorted(groups):
    for v in groups[root_name]:
        vol = piece_volume(v.st_hpoly)
        n_total += 1
        if vol < VOL_LO:
            n_tiny += 1
        elif vol > VOL_HI:
            n_huge += 1
        else:
            survivors.setdefault(root_name, []).append((v, vol))

n_survivors = sum(len(v) for v in survivors.values())
print(f"{n_total} pieces total: {n_tiny} filtered as degenerate/tiny (<{VOL_LO}), "
      f"{n_huge} filtered as huge leftovers (>{VOL_HI}), {n_survivors} left to plot")
for root_name in sorted(survivors):
    vols_here = [f'{vol:.4f}' for _, vol in survivors[root_name]]
    print(f"  {root_name}: {len(survivors[root_name])} pieces, volumes={vols_here}")

135 pieces total: 33 filtered as degenerate/tiny (<0.0001), 18 filtered as huge leftovers (>1.0), 84 left to plot
  v0: 13 pieces, volumes=['0.4946', '0.0164', '0.2866', '0.0259', '0.0040', '0.0881', '0.0033', '0.0053', '0.0018', '0.0539', '0.0500', '0.0454', '0.0002']
  v1: 21 pieces, volumes=['0.0115', '0.0143', '0.0002', '0.0387', '0.0008', '0.8985', '0.0465', '0.8991', '0.0618', '0.0790', '0.0092', '0.0026', '0.0028', '0.0984', '0.0002', '0.0053', '0.0001', '0.0111', '0.0009', '0.0009', '0.0001']
  v2: 20 pieces, volumes=['0.0377', '0.0549', '0.0552', '0.0003', '0.0003', '0.7417', '0.6519', '0.0001', '0.0799', '0.0196', '0.0001', '0.0004', '0.0005', '0.0159', '0.0002', '0.0556', '0.0524', '0.0417', '0.0002', '0.0003']
  v3: 19 pieces, volumes=['0.0042', '0.0673', '0.0009', '0.0008', '0.8981', '0.8980', '0.0647', '0.0556', '0.0530', '0.0285', '0.0053', '0.0966', '0.0034', '0.0308', '0.9585', '0.0017', '0.0002', '0.0675', '0.0010']
  v4: 11 pieces, volumes=['0.0554', '0.0445', '0.050

Every surviving piece gets its own legend entry, grouped by region -- click any entry
to show/hide just that one piece (the legend defaults to grouped-click, which would
toggle a whole region at once; `groupclick="toggleitem"` below overrides that so each
piece is independently selectable).

A dropdown (top right) jumps straight to one region -- "All" restores everything,
"v0".."v4" hides every other region's pieces in one click. It drives the same
`visible` property the legend clicks do, so the two compose: pick a region from the
dropdown, then fine-tune individual pieces from the legend.

**One shared 3D scene, not one subplot per region** -- an earlier version used
`make_subplots` with a separate `scene` per region, but hiding a region's traces
there just leaves its panel sitting empty (the subplot's own axes/grid/title stay,
only the content vanishes), which reads as "nothing happened." With everything in a
single scene, hiding a region's pieces makes them actually disappear from the one
view instead of leaving a dead panel behind. Pieces are colored by region here (all
of a region's pieces share a color) so the "All" view stays legible; the dropdown is
what does the real decluttering.

In [17]:
region_cols = [r for r in sorted(survivors) if survivors[r]]

fig_decomp = go.Figure()

for region_idx, root_name in enumerate(region_cols):
    color = REGION_COLORS[region_idx % len(REGION_COLORS)]
    for i, (v, vol) in enumerate(survivors[root_name]):
        verts = piece_verts(v.st_hpoly)
        trace = hull_mesh3d(
            verts, color, 0.55, f"{root_name} piece {i} (vol={vol:.4f})",
            showlegend=True, legendgroup=root_name,
            legendgrouptitle=root_name if i == 0 else None,
        )
        fig_decomp.add_trace(trace)

# dropdown: pick one region to isolate, or "All" to restore every piece. Drives the
# same per-trace `visible` flag the legend clicks toggle, via one `restyle` per button.
n_traces = len(fig_decomp.data)
trace_regions = [t.legendgroup for t in fig_decomp.data]
dropdown_buttons = [dict(label="All", method="restyle", args=[{"visible": [True] * n_traces}])]
for root_name in region_cols:
    dropdown_buttons.append(dict(
        label=root_name, method="restyle",
        args=[{"visible": [g == root_name for g in trace_regions]}],
    ))

fig_decomp.update_layout(
    title=f"surviving decomposition pieces (volume in [{VOL_LO}, {VOL_HI}]) -- "
          f"drag to rotate, click legend entries or use the dropdown to isolate a region",
    scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="t"),
    legend=dict(groupclick="toggleitem", font=dict(size=9), tracegroupgap=4),
    updatemenus=[dict(
        type="dropdown", buttons=dropdown_buttons, showactive=True,
        x=1.0, xanchor="left", y=1.0, yanchor="top",
    )],
    height=750, width=1100, margin=dict(l=0, r=180, t=60, b=0),
)
fig_decomp